# LlamaIndex · Ask the Cookbook

This repo is eight chapters of notebooks, READMEs and modules, about 400,000 tokens of them, and the question someone new to it asks is never "show me chapter 4" but "*how does the runtime stop an agent that keeps calling the same tool?*" or "*why is CrewAI pinned to an old version?*". The answer is somewhere in a README, a notebook's explanation, or the module that implements it.

So this project builds the assistant that answers those questions **from the repo itself**: it finds the passages, answers from them, and cites them. That is what LlamaIndex is built around: getting documents in, splitting and indexing them, keeping the index current, retrieving well, and answering with sources. This notebook is about what LlamaIndex gives you, and what you have to know about it. One part is built a second time from this repo's own parts: the agent, because LlamaIndex's agent has no way to keep what it sends within a request limit, and that is exactly what chapter 5 is about.

**What it reads**: every README; the prose of every notebook (the markdown cells, which is where the notebooks explain); and every module (where the explained things are implemented). It leaves out notebook code cells and outputs, tests and stand-ins, and this folder, so it cannot find its own answer key.

**What it is tested on**: 15 questions the cookbook answers, each with the file the answer is in, written down before the system existed and checked against the files; three it does not answer; and two that need more than one chapter, for the agent.

| File | What it is |
|---|---|
| `cookbook.py` | the corpus as documents, and the questions with their answer key |
| `ask_index.py` | the system: models, splitting, the index, retrieval, the cited engine, the agent, evaluation |
| `agent_scratch.py` | the agent again, from chapters 2, 3, 5 and 6: chapter 5 chooses what each call is sent |
| `ask_stubs.py` | stand-ins for every model, so all of it runs offline first |
| `ask_checks.py` | what building it found out about LlamaIndex, each reproduced |
| `test_ask.py` | offline tests for all of it |

---
# Part 1 · Build

## The corpus

One LlamaIndex `Document` per file, with its path as its id. The id is what lets the index replace a file's old version when the file changes, rather than adding a second copy.

```python
# cookbook.py
def documents():
    """One `Document` per file. Its id is the file's path, so a changed file replaces its old version."""
    from llama_index.core import Document
    docs = []
    for path in files():
        rel = path.relative_to(REPO).as_posix()
        kind = "notebook" if path.suffix == ".ipynb" else "code" if path.suffix == ".py" else "readme"
        text = _notebook_prose(path) if kind == "notebook" else path.read_text(encoding="utf-8")
        if not text.strip():
            continue
        chapter = rel.split("/")[0] if "/" in rel else "(repo)"
        docs.append(Document(id_=rel, text=text, metadata={"file": rel, "chapter": chapter, "kind": kind}))
    return docs
```

## The models

The answers come from gpt-oss-120b on Groq, through LlamaIndex's `OpenAILike`. LlamaIndex reuses the OpenAI clients it is handed, so that is where the token budget lives: every call, whether an answer or an agent's turn, is counted from the API's own `usage` and refused once the budget is spent. Embeddings come from Gemini's `gemini-embedding-001`, as chapter 4 uses it. LlamaIndex's integration already sends Gemini the two task types chapter 4 measured (`RETRIEVAL_DOCUMENT` going in, `RETRIEVAL_QUERY` for a question), but its default model is a preview, so the model and size are pinned.

```python
# ask_index.py
def chat_model(model=MODEL, budget=None, max_tokens=4096):
    """gpt-oss on Groq, through LlamaIndex's `OpenAILike`, with clients that count and cap what is spent.

    `OpenAILike` reuses the OpenAI clients it is handed, so the clients are where the budget lives:
    every call LlamaIndex makes -- an answer, an agent's turn -- goes through them.
    """
    import openai
    from llama_index.llms.openai_like import OpenAILike
    budget = budget or Budget()
    key = os.environ["GROQ_API_KEY"]
    sync = openai.OpenAI(api_key=key, base_url=GROQ, max_retries=8)
    aio = openai.AsyncOpenAI(api_key=key, base_url=GROQ, max_retries=8)
    real_sync, real_aio = sync.chat.completions.create, aio.chat.completions.create

    def create(*args, **kwargs):
        budget.check()
        response = real_sync(*args, **kwargs)
        budget.charge(response)
        return response

    async def acreate(*args, **kwargs):
        budget.check()
        response = await real_aio(*args, **kwargs)
        budget.charge(response)
        return response

    sync.chat.completions.create, aio.chat.completions.create = create, acreate
    llm = _metered()(model=model, api_base=GROQ, api_key=key, openai_client=sync, async_openai_client=aio,
                     is_chat_model=True, is_function_calling_model=True, context_window=131_072,
                     max_tokens=max_tokens, temperature=0, timeout=120)
    llm._budget = budget
    return llm

def embed_model():
    """Gemini, as chapter 4 uses it. LlamaIndex's integration sends RETRIEVAL_DOCUMENT for what goes into
    the index and RETRIEVAL_QUERY for a question -- the distinction chapter 4 measured -- but its default
    model is a preview one, so the model and the size are pinned."""
    from google.genai import types
    from llama_index.embeddings.google_genai import GoogleGenAIEmbedding
    return GoogleGenAIEmbedding(model_name=EMBED_MODEL, api_key=os.environ["GEMINI_API_KEY"],
                                embedding_config=types.EmbedContentConfig(output_dimensionality=EMBED_DIM),
                                embed_batch_size=50, retries=8, retry_max_seconds=65)
```

## Splitting: prose and code differently

Prose is cut at its markdown headings by `MarkdownNodeParser`, which keeps the heading path as metadata, then to size by `SentenceSplitter`. Code goes to LlamaIndex's `CodeSplitter`, which parses Python with tree-sitter and cuts between definitions rather than mid-function. Both are one transformation, so a saved index has one thing to re-run on a changed file.

Both parsers also leave chunks with almost nothing in them: a class header on its own, or a notebook heading whose section was all code. They are joined onto the next chunk of the same file.

```python
# ask_index.py
def split():
    """One transformation that splits prose and code differently -- one, so a saved index has one thing to
    re-run on a changed file.

    prose   `MarkdownNodeParser` cuts at headings and keeps the heading path as metadata; a section longer
            than 800 tokens is then cut by `SentenceSplitter`
    code    `CodeSplitter` parses the module with tree-sitter and cuts between definitions. Its defaults
            (40 lines, 1,500 characters) cut a module's docstring into fragments -- one of them just `\"\"\"`
            -- so a chunk may run to 80 lines and 4,000 characters
    """
    from llama_index.core.node_parser import CodeSplitter, MarkdownNodeParser, SentenceSplitter
    from llama_index.core.schema import TransformComponent

    prose_parser, cap = MarkdownNodeParser(), SentenceSplitter(chunk_size=800, chunk_overlap=80)
    code_parser = CodeSplitter(language="python", chunk_lines=80, chunk_lines_overlap=10, max_chars=4000)

    class Split(TransformComponent):
        def __call__(self, nodes, **kwargs):
            prose = [n for n in nodes if n.metadata.get("kind") != "code"]
            code = [n for n in nodes if n.metadata.get("kind") == "code"]
            out = joined(cap(prose_parser(prose))) + joined(code_parser(code))
            for node in out:
                # the file and the heading go into the vector: "chapter 4" in a question should find chapter 4
                node.excluded_embed_metadata_keys = ["kind"]
            return out

    return Split()

def joined(nodes, least=25):
    """A chunk with almost nothing in it, joined onto the next chunk of the same file.

    Both parsers make them on this repo: `CodeSplitter` leaves a class's header on its own ("class Runtime:"),
    so its methods lose the class's name; and a notebook heading whose section is code ("## Setup") is a
    section with no prose once the code cells are left out. Alone, each is a chunk that matches nothing
    and still takes a place in the index.
    """
    import re
    out, carry = [], None
    for node in nodes:
        if carry is not None and carry.ref_doc_id == node.ref_doc_id:
            node.set_content(carry.get_content() + "\n" + node.get_content())
            carry = None
        elif carry is not None:
            out.append(carry)
            carry = None
        if len(re.sub(r"\W", "", node.get_content())) < least:
            carry = node
        else:
            out.append(node)
    if carry is not None:
        out.append(carry)
    return out
```

## An index that stays current

A `VectorStoreIndex`, saved to disk. `refresh_ref_docs` is LlamaIndex's way of keeping it current: it compares each file's hash and re-splits and re-embeds only what changed. Two things it does not do are added in `refresh`. It never removes a file that has been deleted, and it saves nothing, so a first build stopped half-way would start again from nothing. Both are reproduced in Part 2.

```python
# ask_index.py
def open_index(path=INDEX_DIR, embed=None):
    """The saved index, or an empty one. The splitter is passed back in on load: a loaded index otherwise
    re-splits a changed file with LlamaIndex's default splitter, not the one it was built with."""
    from llama_index.core import StorageContext, VectorStoreIndex, load_index_from_storage
    embed = embed or embed_model()
    if (Path(path) / "docstore.json").exists():
        storage = StorageContext.from_defaults(persist_dir=str(path))
        return load_index_from_storage(storage, embed_model=embed, transformations=[split()])
    return VectorStoreIndex([], embed_model=embed, transformations=[split()])

def refresh(index, documents, path=INDEX_DIR, every=10, log=print):
    """Bring the index up to date with the files, and save it as it goes.

    LlamaIndex's `refresh_ref_docs` re-splits and re-embeds a document whose hash has changed and inserts a
    new one. Two things it does not do are added here: it never removes a document that is no longer in
    the corpus, so a deleted file would go on being retrieved; and it saves nothing, so a first build
    stopped half-way -- by a rate limit, say -- would start again from nothing. Saving every `every` files
    makes the build resumable, and a resumed build skips every file already in.
    """
    changed = []
    for document in documents:
        if index.docstore.get_document_hash(document.id_) == document.hash:
            continue
        index.refresh_ref_docs([document])
        changed.append(document.id_)
        if path and len(changed) % every == 0:
            index.storage_context.persist(persist_dir=str(path))
            log(f"  {len(changed)} files in, latest {document.id_}")
    present = {d.id_ for d in documents}
    removed = sorted(ref for ref in index.ref_doc_info if ref not in present)
    for ref in removed:
        index.delete_ref_doc(ref, delete_from_docstore=True)
    if path:
        index.storage_context.persist(persist_dir=str(path))
    return {"changed": changed, "removed": removed, "files": len(index.ref_doc_info), "chunks": len(index.docstore.docs)}
```

## Retrieval, with a floor

Vector search and BM25, fused by reciprocal rank, so a question that names an exact function finds it even when its meaning sits elsewhere. In front of both is a **floor**, chapter 4's lesson: a vector index always returns its *k* nearest, however far away they are. Fusion turns scores into ranks, so the floor has to come first, on the raw cosine. The floor is measured, not chosen:

```python
# ask_index.py
# Measured with measure_floor() on the built index, not chosen: the 15 questions the cookbook answers score
# 0.702-0.787 on their best passage, the three it does not 0.539, 0.547 and 0.624, and the floor sits midway
# between 0.624 and 0.702. The 0.624 is "How do I fine-tune Llama 3 on my own data?": the cookbook talks about
# Llama models throughout, so the question is near in words, not in subject. From 18 questions; it moves with
# the corpus and the embedding model.
FLOOR = 0.66
```

```python
# ask_index.py
class CookbookRetriever(BaseRetriever):
    """Vector search and BM25, fused by reciprocal rank -- behind a floor on the vector search.

    Fusion turns scores into ranks, so after it there is no similarity left to set a floor on: a question
    about the capital of Australia still gets five "best" passages. The floor is applied first, on the
    vector search's raw cosine, as chapter 4's `Recall` does: if nothing in the cookbook is near the
    question, no passages are returned and no answer is written. The question is embedded once; the vector
    retriever stores the embedding on the query, and the fused search reuses it.

    BM25 is built from the index's chunks when this is made, so make a new one after a refresh.
    """

    def __init__(self, index, llm, k=5, floor="default"):
        from llama_index.core.retrievers import QueryFusionRetriever
        from llama_index.retrievers.bm25 import BM25Retriever
        self.vector = index.as_retriever(similarity_top_k=2 * k)
        self.bm25 = BM25Retriever.from_defaults(docstore=index.docstore, similarity_top_k=2 * k)
        # num_queries=1: fuse the two searches of the question as asked, without an LLM writing variants.
        # The LLM is still passed: QueryFusionRetriever otherwise reaches for LlamaIndex's global default
        self.fused = QueryFusionRetriever([self.vector, self.bm25], llm=llm, num_queries=1,
                                          mode="reciprocal_rerank", similarity_top_k=k, use_async=False)
        self.floor = FLOOR if floor == "default" else floor
        self.best = None
        super().__init__()

    def _retrieve(self, query_bundle):
        near = self.vector.retrieve(query_bundle)
        self.best = near[0].score if near else None
        if self.floor is not None and (self.best is None or self.best < self.floor):
            return []
        return self.fused.retrieve(query_bundle)
```

## Answers with citations

LlamaIndex's `CitationQueryEngine` numbers the passages it is given and asks the model to cite them as [1], [2]. When the floor turns a question away there are no passages, the engine makes no model call, and the answer says so.

```python
# ask_index.py
def engine(index, llm, k=5, floor="default"):
    """LlamaIndex's `CitationQueryEngine` over the cookbook retriever: sources are numbered, and the answer
    cites them as [1], [2]. It re-cuts each retrieved passage into citation-sized pieces first, so code
    chunks get a citation size large enough to keep a function whole."""
    from llama_index.core.query_engine import CitationQueryEngine
    return CitationQueryEngine.from_args(index, retriever=CookbookRetriever(index, llm, k=k, floor=floor),
                                         llm=llm, citation_chunk_size=1024, citation_chunk_overlap=0)

def ask(query_engine, question):
    """One question: the answer, its numbered sources, and whether the floor turned it away."""
    response = query_engine.query(question)
    sources = [{"n": i + 1, "where": label(s.node)} for i, s in enumerate(response.source_nodes)]
    refused = not response.source_nodes
    return {"question": question, "answer": NOT_COVERED if refused else str(response).strip(),
            "sources": sources, "refused": refused, "best score": query_engine.retriever.best,
            "response": response}
```

## The agent

Some questions need more than one search: "how does chapter 6's board differ from chapter 4's stores?" is two questions and a comparison. A LlamaIndex `FunctionAgent` gets two tools, written as plain functions whose signatures and docstrings become the tool schemas: `search_cookbook`, the same retriever as the engine's, and `open_file`, to read on when a passage stops short. `open_file` reads only files the index holds, whatever path it is given.

What the tools hand back was settled by watching the agent fail, four ways:

- **Size.** Every result stays in the agent's conversation and every call re-sends all of it, and Groq's free tier refuses a single request over 8,000 tokens. With five whole passages a search, both questions died on that limit after three to five calls; LlamaIndex's memory limit does not prevent it (section 8). So a search returns three passages of at most 1,200 characters, and `open_file` at most 100 lines.
- **Where.** Cut to their first 1,200 characters, passages lost the answer: in `case.py`'s main passage, the sentence about a resumed node re-running from its first line starts at character 1,559. So a long passage shows the run of lines nearest the query, and every passage says which lines of its file it is, so `open_file` can go straight there; without that, both agents opened the notebook at the top and never reached its findings.
- **The floor.** A floor on meaning alone turned away an exact name: "double_send" scored 0.644 and came back empty, where BM25 would have matched it. The floor is for the one-shot engine, which has to refuse; the agent's search has none, and the agent judges relevance itself.
- **Going round.** A call it has already made is refused, which is chapter 2's doom-loop rule: LlamaIndex's agent has nothing that notices it going in circles, and once searched "hazard" twice running. And after **eight tool calls** it must answer from what it has: LlamaIndex's `max_iterations` with `early_stopping_method="generate"`, a last call with no tools offered.

```python
# ask_index.py
def excerpt(text, query, size=PASSAGE_CHARS):
    """The run of whole lines, at most `size` characters, that shares the most words with the query.

    Cutting a passage to its first 1,200 characters cut the answer out of it: in case.py's main chunk, the
    sentence about a resumed node re-running from its first line starts at character 1,559. So the window
    that bears on the question is shown instead. Returns (first, last, text), lines counted within the passage.
    """
    import re
    words = set(re.findall(r"[a-z_]{3,}", query.lower()))
    lines = text.splitlines() or [""]
    best = (-1, 0, 1)
    for i in range(len(lines)):
        j, used = i, 0
        while j < len(lines) and used + len(lines[j]) + 1 <= size:
            used += len(lines[j]) + 1
            j += 1
        j = max(j, i + 1)
        score = sum(word in line.lower() for line in lines[i:j] for word in words)
        if score > best[0]:
            best = (score, i, j)
    _, i, j = best
    return i, j - 1, "\n".join(lines[i:j])[:size]

def tool_functions(index, llm, k=SEARCH_K, floor=None):
    """The agent's two tools as plain functions: LlamaIndex's agent and the scratch agent both get these.

    No floor by default. The floor is for the one-shot engine, which must turn away a question nothing is near;
    an agent searching step by step judges relevance itself, and a floor on meaning alone turns away an exact
    name: "double_send" scored 0.644 and came back empty, where BM25 would have matched it.
    """
    retriever = CookbookRetriever(index, llm, k=k, floor=floor)
    readable = {p.relative_to(CB.REPO).as_posix(): p for p in CB.files()}

    # Chapter 2's doom-loop rule, which LlamaIndex's FunctionAgent has no equivalent of: the same call with
    # the same arguments again is refused. Live, on a two-sided question, the agent searched "hazard" twice
    # running and drifted until its budget ran out. One set of calls per `tools()`, so one per question.
    made = set()

    def again(*call):
        key = tuple(" ".join(str(part).lower().split()) for part in call)
        if key in made:
            return ("You already made exactly this call, and what it returned is above. Answer from what you have "
                    "found, or try something different.")
        made.add(key)
        return None

    def search_cookbook(query: str) -> str:
        """Search the cookbook. Returns the best passages, each labelled with its file, section and the lines it
        came from; a long passage shows the part nearest the query. Use open_file with those lines to read more."""
        repeated = again("search", query)
        if repeated:
            return repeated
        nodes = retriever.retrieve(query)
        if not nodes:
            return "Nothing in the cookbook is close to that."
        shown = []
        for hit in nodes:
            text, start = hit.node.get_content(), first_line(hit.node)
            i, j, part = excerpt(text, query)
            where = f", lines {start + i}-{start + j}" if start else ""
            longer = "" if part == text else (
                f"  (part of a longer passage" + (f", lines {start}-{start + text.count(chr(10))}" if start else "") + ")")
            shown.append(f"[{label(hit.node)}{where}]{longer}\n{part}")
        return "\n\n".join(shown)

    # `line_start`/`line_end`: the names gpt-oss reaches for first. Named the other way round, its first call
    # failed on every question and it tried again -- a whole call, and a whole conversation re-sent, wasted
    def open_file(path: str, line_start: int = 1, line_end: int = FILE_LINES) -> str:
        """Read lines of one cookbook file, by the path a search result gave, at most 100 lines at a time.
        Notebooks are read as their prose."""
        if path not in readable:
            # only what is indexed: not .env, not anything outside the repo, whatever the path says
            return f"Not a cookbook file: {path!r}. Use a path exactly as search_cookbook labels it."
        repeated = again("open", path, line_start, line_end)
        if repeated:
            return repeated
        file = readable[path]
        text = CB._notebook_prose(file) if file.suffix == ".ipynb" else file.read_text(encoding="utf-8")
        lines = text.splitlines()
        start = max(1, line_start)
        end = min(len(lines), line_end, start + MAX_FILE_LINES - 1)
        return "\n".join(f"{i:>4}  {lines[i - 1]}" for i in range(start, end + 1)) + \
            (f"\n... ({len(lines)} lines in all)" if end < len(lines) else "")

    return [search_cookbook, open_file]

def agent(index, llm, k=SEARCH_K, floor=None):
    from llama_index.core.agent.workflow import FunctionAgent
    # streaming off: a streamed reply carries no `usage`, and the budget is counted from `usage`
    return FunctionAgent(tools=tools(index, llm, k=k, floor=floor), llm=llm, system_prompt=AGENT_PROMPT,
                         streaming=False, timeout=300, early_stopping_method="generate")

async def run_agent(the_agent, question, max_iterations=MAX_TOOL_CALLS + 1, memory=None):
    """One question to the agent: its answer, and every tool call it made on the way.

    A run stopped part-way -- its budget spent, or LlamaIndex's step limit reached -- still comes back with
    the calls it made and why it stopped, rather than as an exception that takes the record with it.
    """
    from llama_index.core.agent.workflow import ToolCallResult
    handler = the_agent.run(user_msg=question, max_iterations=max_iterations,
                            **({"memory": memory} if memory is not None else {}))
    calls, answer, stopped = [], None, None
    try:
        async for event in handler.stream_events():
            if isinstance(event, ToolCallResult):
                calls.append({"tool": event.tool_name, "args": event.tool_kwargs,
                              "returned chars": len(str(event.tool_output))})
        answer = str(await handler).strip()
    except Exception as error:      # noqa: BLE001 -- a stopped run is reported, not raised
        stopped = f"{type(error).__name__}: {error}"
    return {"question": question, "answer": answer, "stopped": stopped, "calls": calls}
```

## The same agent, from this repo's parts

LlamaIndex's agent sends everything it has found on every call. A model keeps nothing between calls, so its conversation has to be sent each time, and LlamaIndex's agent sends all of it: the question, every call, every result. Nothing between calls chooses or trims, and its memory limit acts only between questions. On Groq's free tier, which refuses a single request over 8,000 tokens, that is a ceiling on how many tools an agent can call.

Chapter 5 is the alternative: context **chosen per call**. The same agent is built on chapter 2's loop, with the same two tool functions over the same index, so only the loop and its context differ. Chapter 5's `ContextSystem` wraps the model's Think step with a `Policy`: at most 5,000 tokens a call, the question pinned, the last two results always kept, and older results kept or dropped by how much they bear on the question. The conversation keeps everything; each *call* gets a chosen part of it.

It gets the same eight tool calls, through chapter 2's own `max_tool_calls` and its graceful finish. That needed one adapter: the runtime finds the Think to finish with by its `finish` method, and chapter 5's `Focus` does not pass one through, so a focused Think would have ended a stopped run on "Stopping: ... partial progress". `FocusedThink` forwards it, with the same chosen context as every other call.

```python
# agent_scratch.py
CONTEXT_TOKENS = 5000      # what one Think call may be sent: 8,000 is the request limit, and tool schemas,
KEEP_LAST = 2              # the system prompt and the model's own output fit in what is left
# Every kind but `tool_result`: Observe passes a result through verbatim, so the observation IS the tool's
# result, and chapter 5 lists the raw result again as its own item. Measured offline: 6,538 tokens of
# observations and 6,413 of the same text again as raw results.
THINK_SEES = ("instruction", "request", "constraint", "message", "observation", "decision", "state",
              "fact", "memory", "summary")
```

```python
# agent_scratch.py
def build(index, model=A.MODEL, context_tokens=CONTEXT_TOKENS, keep_last=KEEP_LAST, floor=None,
          max_tool_calls=A.MAX_TOOL_CALLS):
    """The loop, and the context system that chooses what its Think is sent."""
    from llama_index.core.llms import MockLLM

    from act import Act
    from observe import Observe
    from registry import Registry
    from runtime import Runtime
    from supervisor import Think, verbatim
    from system import ContextSystem, Policy
    from tools import Tool

    from focus import Focus, View, to_messages

    class FocusedThink(Focus):
        """Chapter 5's `Focus`, passing chapter 2's graceful finish through.

        Runtime ends a stopped run by giving the last component with a `finish` one more turn, without
        tools, to answer from what it gathered. `Focus` has no `finish`, so a focused Think is invisible to
        it and a stopped run ends on "Stopping: ... partial progress" instead. Forwarded here with the same
        chosen context as every other call -- the whole conversation would be the request over 8,000 tokens.
        """

        def finish(self, conversation, reason):
            return self.component.finish(View(conversation, to_messages(self.items(conversation))), reason)

    # the retriever's fusion step takes an LLM it never calls (one query, no variants): LlamaIndex's MockLLM
    # says so plainly, and keeps the scratch agent's model calls all on chapter 1's client
    registry = Registry([Tool(f) for f in A.tool_functions(index, MockLLM(), floor=floor)])
    context = ContextSystem()
    think = Think(model=model, tools=registry.tools, allow_fork=False, explain=False, max_tokens=4096)
    policy = Policy(budget=context_tokens, keep_last=keep_last, kinds=THINK_SEES)
    focused = FocusedThink(think, prepare=lambda items, conversation: context.apply(items, conversation, policy))
    runtime = Runtime(loop=[focused,
                            Act(functions=registry.functions, schemas=registry.schemas),
                            Observe(extractors={name: verbatim for name in registry.names})],
                      # chapter 2's tool-call budget: after this many, Think answers from what it has
                      repeat_from=0, max_iterations=36, max_tool_calls=max_tool_calls)
    return runtime, context

def run(index, question, budget, **settings):
    """One question: the answer, every tool call, why it stopped if it did, and what each call was sent."""
    from chat import Conversation
    runtime, context = build(index, **settings)
    conversation = Conversation(system=A.AGENT_PROMPT)
    conversation.messages.append({"role": "user", "content": question})
    answer, stopped = None, None
    with metered(budget):
        try:
            runtime.run(conversation)
            answer = conversation.messages[-1]["content"]
            stopped = conversation.stopped or None
        except A.BudgetExceeded as error:
            stopped = f"BudgetExceeded: {error}"
        except ValueError as error:
            # chapter 5's selection refusing: what must go in -- the question, the last results -- does not
            # fit the budget. Reported like a spent budget, because it is one
            if "Selection can leave things out" not in str(error):
                raise
            stopped = f"selection refused: {error}"
        except Exception as error:      # noqa: BLE001 -- Groq refusing a request too large, reported as a stop
            import openai
            if not isinstance(error, openai.APIStatusError):
                raise
            stopped = f"{type(error).__name__}: {error}"
    calls = [{"tool": m["tool"], "args": m["content"]} for m in conversation.messages
             if m.get("role") == "decision" and m.get("type") == "tool_call"]
    return {"question": question, "answer": answer, "stopped": stopped, "calls": calls,
            "context per call": [str(t) for t in context.traces],
            "usage": {**budget.summary(), "refused request": A.refused_size(stopped)}}
```

## Evaluation

Retrieval is scored by LlamaIndex's `RetrieverEvaluator`: a question *hits* if any chunk of an expected file is among the passages returned, and MRR rewards finding it first. The answers are graded against the answer key: whether each contains the word a right answer must, and cites a file the answer is in. None of this is part of the system: it is the test, run while building it. Judging answers with a model is chapters 7 and 8's subject.

```python
# ask_index.py
def expected_ids(index, files):
    """Every chunk of the files an answer is in: a retrieval hits if it returns any of them."""
    return [node_id for ref, info in index.ref_doc_info.items() if ref in files for node_id in info.node_ids]

async def retrieval_scores(index, retriever, questions=None):
    """Hit rate and MRR per question, by LlamaIndex's `RetrieverEvaluator`."""
    from llama_index.core.evaluation import RetrieverEvaluator
    evaluator = RetrieverEvaluator.from_metric_names(["hit_rate", "mrr"], retriever=retriever)
    rows = []
    for question, files, _ in questions or CB.EVAL:
        result = await evaluator.aevaluate(query=question, expected_ids=expected_ids(index, files))
        rows.append({"question": question, **{k: round(v, 3) for k, v in result.metric_vals_dict.items()}})
    return rows
```

---
# Part 2 · Run and show

In [1]:
import json
import warnings

warnings.filterwarnings("ignore")

import ask_checks as K
import ask_index as A
import ask_stubs as S
import cookbook as CB


def show(result):
    print(json.dumps(result, indent=1, ensure_ascii=False, default=str))

## 1 · Offline

The whole system on stand-ins: the real corpus really split and indexed with a bag-of-words embedding, LlamaIndex's `MockLLM` answering, and its `MockFunctionCallingLLM` scripted to search, open a file and answer, so the agent's tool loop really runs.

In [ ]:
import tempfile

offline = A.open_index(path=tempfile.mkdtemp(), embed=S.WordEmbedding())
print({k: v for k, v in A.refresh(offline, CB.documents(), path=None).items() if k != "changed"})
answered = A.ask(A.engine(offline, S.answering(), floor=None), CB.EVAL[0][0])
print("engine sources:", [s["where"] for s in answered["sources"]])
agent_out = await A.run_agent(A.agent(offline, S.scripted()), "Why does Recall apply its floor first?")
print("agent:", [c["tool"] for c in agent_out["calls"]], "->", agent_out["answer"])

## 2 · The index

Opened from disk and refreshed against the files as they are now. The first build embedded all 115 files (about 15 minutes, saved every ten files); after that, a refresh re-embeds only what changed.

In [ ]:
from collections import Counter

index = A.open_index()
refreshed = A.refresh(index, CB.documents())
print({k: v for k, v in refreshed.items() if k != "changed"}, "| re-embedded:", refreshed["changed"] or "nothing")
chunks = list(index.docstore.docs.values())
print("chunks by kind:", dict(Counter(c.metadata["kind"] for c in chunks)))
sizes = sorted(len(c.text) for c in chunks)
print(f"characters per chunk: median {sizes[len(sizes) // 2]:,}, largest {sizes[-1]:,}")

## 3 · The floor, measured

The best vector score each question gets, for the questions the cookbook answers and the ones it does not. Embeddings only; no model call.

In [ ]:
floor = A.measure_floor(index)
print("answered by the cookbook:", sorted(floor["in scope"].values()))
print("not answered by it:     ", floor["out of scope"])
print("floor:", A.FLOOR)

## 4 · Retrieval quality

The 15 questions against three retrievers, top five passages each. Embeddings only.

In [ ]:
from llama_index.core.retrievers import BaseRetriever


class TopFive(BaseRetriever):
    def __init__(self, inner):
        self.inner = inner
        super().__init__()

    def _retrieve(self, query):
        return self.inner.retrieve(query)[:5]


cookbook_retriever = A.CookbookRetriever(index, S.answering(), floor=None)
retrieval = {}
for name, retriever in [("vector", TopFive(cookbook_retriever.vector)), ("BM25", TopFive(cookbook_retriever.bm25)),
                        ("fused", cookbook_retriever)]:
    rows = await A.retrieval_scores(index, retriever)
    retrieval[name] = rows
    print(f"{name:7} hit@5 {sum(r['hit_rate'] for r in rows) / len(rows):.2f}   MRR {sum(r['mrr'] for r in rows) / len(rows):.3f}"
          f"   missed: {[i + 1 for i, r in enumerate(rows) if not r['hit_rate']]}")
for i in sorted({i for rows in retrieval.values() for i, r in enumerate(rows) if not r["hit_rate"]}):
    print(f"  Q{i + 1}: {CB.EVAL[i][0]}")

## 5 · Answers, live

Every question through the cited engine, inside a token budget: the answers and their sources, saved. They are graded in the next cell, from what was saved.

In [ ]:
llm = A.chat_model(budget=A.Budget(90_000, "engine"))
answering = A.engine(index, llm)
answers = []
for question in [q for q, _, _ in CB.EVAL] + CB.OUT_OF_SCOPE:
    a = A.ask(answering, question)
    answers.append({"question": question, "answer": a["answer"], "sources": [s["where"] for s in a["sources"]],
                    "refused": a["refused"], "best score": a["best score"]})
    shown = a["answer"] if len(a["answer"]) <= 400 else a["answer"][:400] + "..."
    print(f"--- {question}\n{shown}\n    sources: {sorted({s.split(' § ')[0] for s in answers[-1]['sources']})}\n")
A.save("answers", {"answers": answers, "usage": llm.budget.summary()})
print(llm.budget.summary())

**Graded** against the answer key: does each answer contain the word a right answer must, and does it cite a file the answer is in? From the saved answers, with no model call, so a correction to the grading never means generating the answers again.

In [ ]:
saved = {a["question"]: a for a in A.load("answers")["answers"]}
right = 0
for question, files, word in CB.EVAL:
    a = saved[question]
    has_word = CB.mentions(a["answer"], word)
    cites = bool({s.split(" § ")[0] for s in a["sources"]} & files)
    right += has_word and cites
    print(f"{'ok ' if has_word and cites else 'NO '}{question}" + ("" if has_word else f"   (no '{word}')")
          + ("" if cites else f"   (cites none of {sorted(files)})"))
turned_away = sum(saved[q]["refused"] for q in CB.OUT_OF_SCOPE)
print(f"\n{right} of {len(CB.EVAL)} have the word and cite an answer file; "
      f"{turned_away} of {len(CB.OUT_OF_SCOPE)} out-of-scope questions turned away")

## 6 · The agent, live

Two questions that need more than one place in the repo. What to read: the tool calls it made, and whether the answer cites both sides.

In [ ]:
agent_runs = []
for question in CB.CROSS_CHAPTER:
    agent_llm = A.chat_model(budget=A.Budget(45_000, "agent"))      # a budget per question
    run = await A.run_agent(A.agent(index, agent_llm), question)
    run["usage"] = agent_llm.budget.summary()
    agent_runs.append(run)
    print(f"--- {question}")
    for call in run["calls"]:
        print(f"    {call['tool']}({call['args']})  -> {call['returned chars']:,} characters back")
    print(f"    stopped: {run['stopped']}" if run["stopped"] else run["answer"][:1500])
    print("   ", run["usage"], "\n")
A.save("agent", {"runs": agent_runs})

## 7 · The same agent, from this repo's parts, live

The same two questions, the same tools, the same 45,000-token budget per question; chapter 5 choosing what each call is sent.

In [ ]:
import agent_scratch as X

scratch_runs = []
for question in CB.CROSS_CHAPTER:
    run = X.run(index, question, A.Budget(45_000, "scratch agent"))
    scratch_runs.append(run)
    print(f"--- {question}")
    for call in run["calls"]:
        print(f"    {call['tool']}({call['args']})")
    print(f"    stopped: {run['stopped']}" if run["stopped"] else run["answer"][:1500])
    print("   ", run["usage"], "\n")
A.save("agent_scratch", {"runs": scratch_runs})

**Side by side**, from the saved runs:

In [2]:
llama, scratch = A.load("agent")["runs"], A.load("agent_scratch")["runs"]
print(f"{'':58}{'LlamaIndex agent':>22}{'repo agent + chapter 5':>26}")
for l, s in zip(llama, scratch):
    print(l["question"][:56])
    for label, value in [("answered", lambda r: "yes" if r["answer"] else "no: " + r["stopped"].split(":")[0]),
                         ("tool calls", lambda r: len(r["calls"])),
                         ("largest call (tokens in + out)", lambda r: f"{r['usage']['largest call (in + out)']:,}"),
                         ("a request refused as too large", lambda r: f"{A.refused_size(r['stopped']):,}"
                          if A.refused_size(r["stopped"]) else "-"),
                         ("tokens, all calls", lambda r: f"{r['usage']['total']:,}")]:
        print(f"    {label:54}{str(value(l)):>22}{str(value(s)):>26}")

                                                                LlamaIndex agent    repo agent + chapter 5
How is chapter 6's shared board different from chapter 4
    answered                                                                 yes                       yes
    tool calls                                                                 5                         8
    largest call (tokens in + out)                                         7,599                     5,554
    a request refused as too large                                             -                         -
    tokens, all calls                                                     20,078                    32,844
In chapter 8's LangGraph project, compare how the LangGr
    answered                                                  no: APIStatusError                       yes
    tool calls                                                                 7                         7
    largest call (tokens in + 

## 8 · What LlamaIndex does that you have to know

All offline but the last.

**No model configured.** What LlamaIndex reaches for when it is not told:

In [ ]:
show(K.default_models())

**`CodeSplitter`'s defaults**, on one of this repo's modules, against the settings used here:

In [ ]:
show(K.code_splitter())

**A saved index, loaded back.** Load it without passing its splitter, change a module, refresh. What is the changed file split with?

In [ ]:
show(K.loaded_splitter())

**A deleted file.** Three modules indexed, one deleted, the index refreshed with `refresh_ref_docs` and with `ask_index.refresh`, then asked about the deleted one:

In [ ]:
show(K.deleted_file())

**A first build stopped half-way**, by an embedding provider failing after 300 texts, then run again:

In [ ]:
show(K.resumable())

**A floor after fusion.** The same question's scores from the vector search and after reciprocal-rank fusion:

In [ ]:
show(K.fused_scores())

**Citation chunks.** The same five passages through `CitationQueryEngine` at its default citation size and at the one used here:

In [ ]:
show(K.citation_chunks())

**A memory limit on the agent.** LlamaIndex's `Memory(token_limit=...)` drops old messages when a conversation grows past it. The agent searches six times, under a limit of a million tokens, of 6,000 and of 1,500; what it sends on each call:

In [ ]:
show(await K.agent_memory_limit())

**Counting tokens** (live, one question): LlamaIndex's `TokenCountingHandler`, which counts with a tokenizer, against the API's own `usage`:

In [ ]:
counted = K.token_counts(index, A.chat_model(budget=A.Budget(15_000, "count")), CB.EVAL[0][0])
show(counted)
A.save("token_counts", counted)

## Findings

**The one-shot engine answered all 15 questions and turned away all 3 it should not answer.** Each answer has the word a right answer needs and cites a file the answer is in, for 48,093 tokens over 15 model calls. The three off-topic questions cost nothing: the floor turns them away before any model is called. Building this took little code. LlamaIndex supplies the splitters, the vector store, fusion, the cited engine, the agent and the retrieval evaluator.

**The floor works, with a thin margin.** The lowest-scoring question the cookbook answers scored 0.702, and the highest-scoring one it does not answer scored 0.624 ("How do I fine-tune Llama 3?", which is close to the cookbook's subject). The floor of 0.66 sits between them. That was measured on 18 questions, so a near-topic question could fall on either side. The floor has to be applied to the raw vector score: after fusion, scores are rank arithmetic (0.03), and a 0.6 floor drops everything.

**On these questions, vector search alone ranked best.** All three retrievers found an answer file in their top five for every question. Vector search ranked it highest (MRR 0.956), against 0.786 for BM25 and 0.822 for the two fused. Fusion pays off on exact names. `double_send` scores below the floor on meaning alone, while BM25 matches the word. These 15 questions are plain English, so they do not test that.

**Both agents answered the first question correctly. On the second, only the one with chosen context got to an answer.**

| | LlamaIndex agent | this repo's parts + chapter 5 |
|---|---|---|
| board vs. stores: answered | yes, 5 tool calls, 20,078 tokens | yes, 8 tool calls (the cap), 32,844 tokens |
| LangGraph vs. case file: answered | **no**: the 8th request was 8,172 tokens, over Groq's 8,000 | yes, 7 tool calls, 29,871 tokens |
| tokens sent per call, second question | 529 → 1,226 → 2,217 → … → 6,723, then refused | rises to about 4,300 of context, then stays there |

LlamaIndex's agent sends everything it has found on every call. The request grows with each search until one is too large, and here that happened before the eight-call cap. Its memory limit does not trim within a run (section 8). The repo's agent kept up to 16,000 tokens of findings in its conversation, and chapter 5 sent each call about 4,300 tokens of it. No call grows with the run. That is not cheaper overall: on the first question it made more calls and spent more. What it buys is that a long search cannot make a request too large. When the scratch agent hit the eight-call cap, chapter 2's graceful finish had it answer from what it had found.

**A well-cited answer can still put facts on the wrong side.** The scratch agent's comparison was checked line by line against the LangGraph notebook. It got the four hazards right: a resumed node re-runs from its first line, answers are matched by position, anyone can answer, and durability stops at the workflow. It also described the case file correctly: a `Wait` ends the step, each answer goes in its approver's field, the owner check applies, and every write is logged. Its line citations are real. It also made four mistakes:

- It says a LangGraph node returns a `Wait`, but that is the case file's.
- It lists parallelism, streaming and rewinding as missing from LangGraph. They are what LangGraph has and the case file lacks, and its own case-file column says so.
- It shows `case.py`'s `_go` labelled as `ap_graph.py`.
- It gives durability as LangGraph's problem, where the notebook says it applies to both.

Retrieval scores and a must-contain word cannot catch this. Claim-by-claim evaluation can, and that is what the LangSmith and LangFuse projects do with this assistant.

**What you have to know about LlamaIndex**, all reproduced in section 8:

- With no model set, it reaches for OpenAI, and it fails only when first used.
- `CodeSplitter`'s defaults cut a 110-line module into eight chunks, two of them a docstring's quotes alone.
- An index loaded from disk forgets its splitter. Pass it again, or changed files are re-split as prose and cut mid-definition.
- `refresh_ref_docs` never removes a deleted file. The deleted file stays indexed and keeps being retrieved.
- A first build resumes only if it was saved along the way.
- `citation_chunk_size=512` splits five passages into nine numbered sources.
- `Memory(token_limit=...)` does not trim inside one run.
- `TokenCountingHandler` matched the API's own count exactly once it was on `Settings.callback_manager`, before the engine was built.